In [11]:
import pandas as pd
import numpy as np


In [12]:
pd.read_csv('data/Training.csv').tail()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis,Unnamed: 133
4915,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,(vertigo) Paroymsal Positional Vertigo,NaN
4916,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,Acne,NaN
4917,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Urinary tract infection,NaN
4918,0,1,0,0,0,0,1,0,0,0,...,0,1,1,1,1,0,0,0,Psoriasis,NaN
4919,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,Impetigo,NaN


In [13]:
import pandas as pd
import numpy as np
import math

class SmartBayesianAI:
    def __init__(self):
        self.priors = {}
        self.likelihoods = {}
        self.symptoms = []
        self.maladies = []
        self.current_probs = {}

    def fit(self, csv_path):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        target_col = 'prognosis'
        if 'Unnamed: 133' in df.columns: df = df.drop('Unnamed: 133', axis=1) # Nettoyage Kaggle
        
        self.maladies = df[target_col].unique()
        self.symptoms = [c for c in df.columns if c != target_col]
        total_rows = len(df)

        for maladie in self.maladies:
            df_m = df[df[target_col] == maladie]
            self.priors[maladie] = len(df_m) / total_rows
            self.likelihoods[maladie] = {s: (df_m[s].sum() + 1) / (len(df_m) + 2) for s in self.symptoms}
        
        self.current_probs = self.priors.copy()

    def calculate_entropy(self, probs):
        """ Calcule l'incertitude actuelle du système """
        return -sum(p * math.log(p + 1e-9) for p in probs.values())

    def get_best_symptom(self, excluded_symptoms):
        """ Trouve le symptôme qui réduit le plus l'entropie attendue """
        best_symptom = None
        min_expected_entropy = float('inf')

        for s in self.symptoms:
            if s in excluded_symptoms: continue

            # Probabilité que le symptôme soit présent : P(S=1) = Σ P(S=1|M) * P(M)
            p_s1 = sum(self.likelihoods[m][s] * self.current_probs[m] for m in self.maladies)
            p_s0 = 1 - p_s1

            # Calcul de l'entropie si S=1 et si S=0 (simplifié)
            # Pour rester performant, on peut aussi utiliser la variance des likelihoods
            variance = np.var([self.likelihoods[m][s] for m in self.maladies])
            
            if variance > 0: # On veut le symptôme qui divise le plus les maladies
                expected_entropy = -variance 
                if expected_entropy < min_expected_entropy:
                    min_expected_entropy = expected_entropy
                    best_symptom = s
        
        return best_symptom

    def update_probs(self, symptom, observed_value):
        """ Applique le théorème de Bayes suite à une réponse (1 ou 0) """
        new_probs = {}
        for m in self.maladies:
            p_s_given_m = self.likelihoods[m][symptom]
            likelihood = p_s_given_m if observed_value == 1 else (1 - p_s_given_m)
            new_probs[m] = self.current_probs[m] * likelihood
        
        # Normalisation
        total = sum(new_probs.values())
        self.current_probs = {m: p / total for m, p in new_probs.items()}

    def run_interactive_diagnostic(self):
        asked = []
        print("\n--- Diagnostic Intelligent (Bayes) ---")
        
        for i in range(10): # Limite à 10 questions maximum
            best_s = self.get_best_symptom(asked)
            if not best_s: break
            
            ans = input(f"Avez-vous le symptôme '{best_s.replace('_', ' ')}' ? (o/n) : ")
            val = 1 if ans.lower() == 'o' else 0
            
            self.update_probs(best_s, val)
            asked.append(best_s)
            
            # Tri et affichage du top
            top = sorted(self.current_probs.items(), key=lambda x: x[1], reverse=True)[0]
            if top[1] > 0.95: # Seuil de confiance à 95%
                print(f"\n✅ Diagnostic probable à {top[1]*100:.1f}% : {top[0]}")
                return
            
            print(f"Probabilité actuelle : {top[0]} ({top[1]*100:.1f}%)")

# Lancement
ai = SmartBayesianAI()
ai.fit("data/Training.csv")
ai.run_interactive_diagnostic()


--- Diagnostic Intelligent (Bayes) ---
Probabilité actuelle : Typhoid (6.1%)
Probabilité actuelle : Typhoid (14.6%)
Probabilité actuelle : Hepatitis D (42.2%)
Probabilité actuelle : Hypoglycemia (82.2%)
Probabilité actuelle : Jaundice (28.8%)
Probabilité actuelle : Hypoglycemia (84.4%)
Probabilité actuelle : Typhoid (29.4%)
Probabilité actuelle : Typhoid (44.4%)
Probabilité actuelle : Typhoid (44.5%)
Probabilité actuelle : Hypoglycemia (80.2%)
